# 05 — Políticas base y capacidad operativa V2

Este notebook evalúa las políticas **Aleatoria** y **Reglas de riesgo** bajo el mismo entorno, cohorte, horizonte, capacidad y semillas. La salida distingue la acción propuesta por el algoritmo de la acción ejecutada después de aplicar la restricción de recursos.

> El entorno es una simulación metodológica. Sus resultados no constituyen recomendaciones clínicas individuales.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = 'feature/entorno-capacidad-v2'
PROJECT_ROOT = Path('/content/pida-rl-datos-publicos')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'gymnasium', 'pandas'])
if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)])
else:
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH])

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Commit evaluado:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Configuración del experimento

Modifica `CAPACIDAD_MENSUAL` para representar los recursos disponibles en cada mes. Los costos son supuestos operativos explícitos y ajustables, no costos monetarios ni clínicos observados.

In [ ]:
CAPACIDAD_MENSUAL = 100
COSTOS_ACCION = {0: 0, 1: 1, 2: 3, 3: 5}
HORIZONTE_MESES = 12
SEMILLAS_EVALUACION = (42, 123, 2026)

NOMBRES_ACCION = {
    0: 'Sin contacto',
    1: 'Mensaje',
    2: 'Llamada',
    3: 'Teleorientación',
}

assert CAPACIDAD_MENSUAL >= 0
assert set(COSTOS_ACCION) == {0, 1, 2, 3}
print('Capacidad mensual:', CAPACIDAD_MENSUAL)
print('Costos:', COSTOS_ACCION)

## Cohorte de evaluación

Se carga la cohorte preparada por el notebook 03 desde Google Drive. No se generan pacientes nuevos en este notebook.

In [ ]:
from google.colab import drive
import numpy as np
import pandas as pd

drive.mount('/content/drive', force_remount=False)
PERSISTENT_ROOT = Path('/content/drive/MyDrive/PIDA-RL-Diabetes')
COHORT_PATH = PERSISTENT_ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
OUTPUT_DIR = PERSISTENT_ROOT / 'results' / 'capacidad_v2' / 'politicas_base'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not COHORT_PATH.is_file():
    raise FileNotFoundError(f'No existe {COHORT_PATH}. Ejecuta primero el notebook 03.')

cohorte = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})
columnas_requeridas = [
    'FOLIO_INT', 'edad', 'sexo', 'anios_con_diabetes',
    'insulina_diaria', 'depresion', 'consultas_control_12m',
    'num_hospitalizaciones', 'num_complicaciones',
    'score_riesgo', 'categoria_riesgo',
]
faltantes = [c for c in columnas_requeridas if c not in cohorte.columns]
if faltantes:
    raise KeyError('Faltan columnas: ' + ', '.join(faltantes))
if cohorte['FOLIO_INT'].duplicated().any():
    raise ValueError('FOLIO_INT contiene identificadores duplicados.')
if not cohorte['categoria_riesgo'].isin(['bajo', 'medio', 'alto']).all():
    raise ValueError('Se encontraron categorías de riesgo no válidas.')

print('Cohorte:', cohorte.shape)
print('Ruta de resultados:', OUTPUT_DIR)
display(cohorte.head())

## Evaluación común

Ambas políticas se ejecutan con las mismas semillas. Para cada política se crea una instancia nueva del entorno con idéntica configuración.

In [ ]:
from src.environment.diabetes_followup_env import DiabetesFollowUpEnv
from src.evaluation.evaluate_policies import evaluar_politicas
from src.evaluation.reporting import construir_reporte, exportar_reporte_csv
from src.policies.baselines import POLITICAS_BASE

def crear_entorno():
    return DiabetesFollowUpEnv(
        cohorte_df=cohorte,
        capacidad_mensual=CAPACIDAD_MENSUAL,
        costos_accion=COSTOS_ACCION,
        horizonte=HORIZONTE_MESES,
    )

resultados = evaluar_politicas(
    env_factory=crear_entorno,
    politicas=POLITICAS_BASE,
    semillas=SEMILLAS_EVALUACION,
)
reporte = construir_reporte(resultados)
rutas = exportar_reporte_csv(reporte, OUTPUT_DIR)
print('Archivos exportados:')
for nombre, ruta in rutas.items():
    print(f'- {nombre}: {ruta}')

## Validaciones de capacidad

Estas comprobaciones detienen el notebook si una política supera la capacidad o si el registro queda incompleto.

In [ ]:
detalle = reporte['detalle']
assert (detalle['recursos_despues'] >= 0).all(), 'Hay recursos negativos.'
assert (detalle['recursos_usados_mes'] <= detalle['capacidad_mensual']).all(), 'Se excedió la capacidad.'
assert detalle['accion_propuesta'].isin([0, 1, 2, 3]).all()
assert detalle['accion_ejecutada'].isin([0, 1, 2, 3]).all()
filas_esperadas = len(cohorte) * HORIZONTE_MESES * len(SEMILLAS_EVALUACION) * len(POLITICAS_BASE)
assert len(detalle) == filas_esperadas, f'Se esperaban {filas_esperadas} decisiones y se obtuvieron {len(detalle)}.'
print('Validaciones correctas:', len(detalle), 'decisiones registradas.')

## Resultados para revisión

La primera tabla compara métricas por algoritmo. Las tablas siguientes muestran las clasificaciones propuestas y ejecutadas; su diferencia representa ajustes causados por la capacidad disponible.

In [ ]:
display(reporte['resumen_algoritmos'].round(4))
display(reporte['clasificaciones_propuestas'].round(2))
display(reporte['clasificaciones_ejecutadas'].round(2))
display(detalle.head(20))

## Interpretación

- **Acción propuesta:** clasificación emitida por la política.
- **Acción ejecutada:** clasificación permitida después de descontar recursos.
- **Ajuste de capacidad:** la propuesta no cabía y se redujo a la intervención viable de mayor intensidad.
- **Cobertura de alto riesgo:** proporción de decisiones de riesgo alto con alguna acción ejecutada.
- **Utilización de capacidad:** recursos consumidos divididos entre recursos disponibles durante el episodio.

Los resultados describen comportamiento dentro del simulador y no efectividad clínica observada.